In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/pill_detection'
DATA_DIR = f'{BASE}/data'
TRAIN_IMG_DIR = f'{DATA_DIR}/train_images'
TRAIN_ANN_DIR = f'{DATA_DIR}/train_annotations'
TEST_IMG_DIR = f'{DATA_DIR}/test_images'
OUTPUT_DIR = f'{BASE}/outputs'
YOLO_DIR = f'{BASE}/yolo_data'
yaml_path = f'{YOLO_DIR}/data.yaml'

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

!pip install ultralytics ensemble-boxes -q

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 34.7 MB/s eta 0:00:00


In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11m.pt')

results = model.train(
    data=yaml_path,
    epochs=100,
    imgsz=1024,
    batch=4,           # T4 기준 m 모델은 4 권장
    project=f'{BASE}/runs',
    name='yolo11m_exp',
    optimizer='AdamW',
    lr0=0.001,
    warmup_epochs=3,
    cos_lr=True,
    # 증강
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=15,
    flipud=0.3,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.1,
    # 클래스 불균형 대응
    cls=1.5,
    patience=20,
    save=True,
    device=0
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.45 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=1.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/drive/MyDrive/pill_detection/yolo_data/data.yaml, degrees=15, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.3, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, 

In [ ]:
from ultralytics import YOLO
import os, csv

def predict_with_tta(model_path, model_name):
    model = YOLO(model_path)
    test_imgs = sorted([
        f for f in os.listdir(TEST_IMG_DIR) if f.endswith('.png')
    ])

    rows = []
    ann_id = 1

    for img_file in test_imgs:
        image_id = int(img_file.replace('.png', ''))
        img_path = os.path.join(TEST_IMG_DIR, img_file)

        # TTA: augment=True
        results = model.predict(
            img_path,
            conf=0.25,
            iou=0.5,
            augment=True,   # TTA
            verbose=False
        )

        for r in results:
            if r.boxes is None:
                continue
            for box in r.boxes:
                x1, y1, x2, y2 = box.xyxy[0].tolist()
                w, h = x2 - x1, y2 - y1
                score = float(box.conf[0])
                cat_id = int(box.cls[0]) + 1

                rows.append([ann_id, image_id, cat_id,
                              round(x1,2), round(y1,2),
                              round(w,2), round(h,2),
                              round(score,4)])
                ann_id += 1

    out_path = f'{OUTPUT_DIR}/pred_{model_name}.csv'
    with open(out_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['annotation_id','image_id','category_id',
                         'bbox_x','bbox_y','bbox_w','bbox_h','score'])
        writer.writerows(rows)

    print(f"[{model_name}] {len(rows)} rows saved → {out_path}")
    return out_path

# 기존 모델들 예측
pred_8s  = predict_with_tta(f'{BASE}/runs/baseline_yolov8s/weights/best.pt',  'v8s')
pred_8m  = predict_with_tta(f'{BASE}/runs/yolov8m_exp/weights/best.pt',       'v8m')
pred_11s = predict_with_tta(f'{BASE}/runs/yolo11s_exp/weights/best.pt',       'v11s')
# yolo11m 학습 완료 후:
# pred_11m = predict_with_tta(f'{BASE}/runs/yolo11m_exp/weights/best.pt', 'v11m')

[v8s] 3509 rows saved → /content/drive/MyDrive/pill_detection/outputs/pred_v8s.csv
[v8m] 3491 rows saved → /content/drive/MyDrive/pill_detection/outputs/pred_v8m.csv
[v11s] 3375 rows saved → /content/drive/MyDrive/pill_detection/outputs/pred_v11s.csv


In [ ]:
pred_11m = predict_with_tta(f'{BASE}/runs/yolo11m_exp/weights/best.pt', 'v11m')

[v11m] 3536 rows saved → /content/drive/MyDrive/pill_detection/outputs/pred_v11m.csv


In [ ]:
from ensemble_boxes import weighted_boxes_fusion
import pandas as pd
import numpy as np

def load_preds(csv_path):
    return pd.read_csv(csv_path)

def run_wbf_ensemble(pred_paths, weights, iou_thr=0.55, skip_box_thr=0.3,
                     out_name='ensemble_tuned'):
    """
    pred_paths: CSV 경로 리스트
    weights: 각 모델 가중치 리스트 (합이 1일 필요 없음)
    """
    dfs = [load_preds(p) for p in pred_paths]

    # 이미지 ID 전체 수집
    all_image_ids = sorted(set().union(*[set(df['image_id']) for df in dfs]))

    # 이미지 크기 (EDA 결과: 976×1280)
    IMG_W, IMG_H = 976, 1280

    rows = []
    ann_id = 1

    for img_id in all_image_ids:
        boxes_list, scores_list, labels_list = [], [], []

        for df in dfs:
            sub = df[df['image_id'] == img_id]
            if len(sub) == 0:
                boxes_list.append([])
                scores_list.append([])
                labels_list.append([])
                continue

            # WBF는 [0,1] 정규화 좌표 필요
            bboxes = []
            for _, row in sub.iterrows():
                x1 = row['bbox_x'] / IMG_W
                y1 = row['bbox_y'] / IMG_H
                x2 = (row['bbox_x'] + row['bbox_w']) / IMG_W
                y2 = (row['bbox_y'] + row['bbox_h']) / IMG_H
                # 클리핑
                bboxes.append([
                    max(0, min(1, x1)), max(0, min(1, y1)),
                    max(0, min(1, x2)), max(0, min(1, y2))
                ])

            boxes_list.append(bboxes)
            scores_list.append(sub['score'].tolist())
            labels_list.append(sub['category_id'].tolist())

        # WBF 실행
        boxes, scores, labels = weighted_boxes_fusion(
            boxes_list, scores_list, labels_list,
            weights=weights,
            iou_thr=iou_thr,
            skip_box_thr=skip_box_thr
        )

        for box, score, label in zip(boxes, scores, labels):
            x1 = box[0] * IMG_W
            y1 = box[1] * IMG_H
            w  = (box[2] - box[0]) * IMG_W
            h  = (box[3] - box[1]) * IMG_H
            rows.append([ann_id, img_id, int(label),
                         round(x1,2), round(y1,2),
                         round(w,2), round(h,2),
                         round(float(score),4)])
            ann_id += 1

    out_path = f'{OUTPUT_DIR}/{out_name}.csv'
    with open(out_path, 'w', newline='') as f:
        import csv
        writer = csv.writer(f)
        writer.writerow(['annotation_id','image_id','category_id',
                         'bbox_x','bbox_y','bbox_w','bbox_h','score'])
        writer.writerows(rows)

    print(f"[{out_name}] {len(rows)} rows → {out_path}")
    return out_path

# 가중치 실험 조합 (v8s, v8m, v11s)
experiments = {
    'equal':      [1.0, 1.0, 1.0],
    'boost_11s':  [0.8, 1.0, 1.2],
    'boost_8m':   [0.8, 1.2, 1.0],
    'heavy_big':  [0.5, 1.5, 1.5],
}
pred_paths_3 = [pred_8s, pred_8m, pred_11s]

for name, w in experiments.items():
    run_wbf_ensemble(pred_paths_3, weights=w,
                     iou_thr=0.55, skip_box_thr=0.3,
                     out_name=f'wbf_{name}')

# 4모델 앙상블 추가
pred_paths_4 = [pred_8s, pred_8m, pred_11s, pred_11m]
run_wbf_ensemble(pred_paths_4, weights=[0.8, 1.0, 1.0, 1.2],
                 out_name='wbf_4models')

[wbf_equal] 4044 rows → /content/drive/MyDrive/pill_detection/outputs/wbf_equal.csv
[wbf_boost_11s] 4044 rows → /content/drive/MyDrive/pill_detection/outputs/wbf_boost_11s.csv
[wbf_boost_8m] 4044 rows → /content/drive/MyDrive/pill_detection/outputs/wbf_boost_8m.csv
[wbf_heavy_big] 4044 rows → /content/drive/MyDrive/pill_detection/outputs/wbf_heavy_big.csv
[wbf_4models] 4308 rows → /content/drive/MyDrive/pill_detection/outputs/wbf_4models.csv


'/content/drive/MyDrive/pill_detection/outputs/wbf_4models.csv'

In [ ]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/pill_detection/outputs/wbf_4models.csv')
print(df.head(10))
print('\ncategory_id 최솟값:', df['category_id'].min())
print('score 범위:', df['score'].min(), '~', df['score'].max())
print('bbox_x 범위:', df['bbox_x'].min(), '~', df['bbox_x'].max())

   annotation_id  image_id  category_id  bbox_x  bbox_y  bbox_w  bbox_h  \
0              1         1           39  595.02  676.52  259.64  478.15   
1              2         1            1  157.66  251.31  202.41  124.27   
2              3         1           17  559.87   71.20  393.06  406.40   
3              4         1           42  170.45  745.15  181.66  291.55   
4              5         1           37  558.48   71.59  391.42  405.78   
5              6         1           54  174.84  742.36  176.49  288.63   
6              7         3           39  568.83  629.10  262.51  484.73   
7              8         3            1  139.78  241.29  199.09  127.73   
8              9         3           17  530.68   63.06  382.81  401.15   
9             10         3           42  140.50  706.79  181.64  294.84   

    score  
0  0.9867  
1  0.9616  
2  0.9033  
3  0.3493  
4  0.1418  
5  0.1338  
6  0.9864  
7  0.9647  
8  0.9333  
9  0.1962  

category_id 최솟값: 1
score 범위: 0.0601 ~ 0.9

In [ ]:
df_old = pd.read_csv('/content/drive/MyDrive/pill_detection/outputs/submission_ensemble_all.csv')
print("=== 어제 파일 ===")
print(df_old.head(5))
print('컬럼:', df_old.columns.tolist())

df_new = pd.read_csv('/content/drive/MyDrive/pill_detection/outputs/wbf_4models.csv')
print("\n=== 오늘 파일 ===")
print(df_new.head(5))
print('컬럼:', df_new.columns.tolist())

=== 어제 파일 ===
   annotation_id  image_id  category_id  bbox_x  bbox_y  bbox_w  bbox_h  \
0              1         1        27926  598.09  676.30  256.76  477.91   
1              2         1         1900  157.85  252.27  202.45  123.97   
2              3         1        16551  556.67   72.33  395.74  406.48   
3              4         1        29345  173.07  743.98  179.00  291.25   
4              5         1        27733  555.98   73.12  397.02  408.91   

    score  
0  0.9790  
1  0.9657  
2  0.8942  
3  0.6190  
4  0.2624  
컬럼: ['annotation_id', 'image_id', 'category_id', 'bbox_x', 'bbox_y', 'bbox_w', 'bbox_h', 'score']

=== 오늘 파일 ===
   annotation_id  image_id  category_id  bbox_x  bbox_y  bbox_w  bbox_h  \
0              1         1           39  595.02  676.52  259.64  478.15   
1              2         1            1  157.66  251.31  202.41  124.27   
2              3         1           17  559.87   71.20  393.06  406.40   
3              4         1           42  170.45  7

In [ ]:
import os, json, glob

# 1. YOLO 클래스 순서대로 dl_idx 매핑 만들기
json_files = glob.glob(f'{TRAIN_ANN_DIR}/**/*.json', recursive=True)

# data.yaml에서 클래스 순서 확인
import yaml
with open(yaml_path, 'r') as f:
    data_yaml = yaml.safe_load(f)

class_names = data_yaml['names']  # YOLO 인덱스 순서대로 된 클래스명 리스트
print("클래스 수:", len(class_names))
print("첫 5개:", class_names[:5])

# 2. JSON에서 dl_name → dl_idx 매핑
name_to_dlid = {}
for jf in json_files:
    with open(jf, 'r', encoding='utf-8') as f:
        data = json.load(f)
    for img in data['images']:
        name_to_dlid[img['dl_name']] = int(img['dl_idx'])

# 3. YOLO 인덱스 → category_id 변환 딕셔너리
idx_to_catid = {}
for i, name in enumerate(class_names):
    if name in name_to_dlid:
        idx_to_catid[i] = name_to_dlid[name]
    else:
        print(f"매핑 못 찾음: {name}")

print("매핑 완료:", len(idx_to_catid), "개")
print("샘플:", list(idx_to_catid.items())[:5])

클래스 수: 56
첫 5개: ['보령부스파정 5mg', '뮤테란캡슐 100mg', '일양하이트린정 2mg', '기넥신에프정(은행엽엑스)(수출용)', '무코스타정(레바미피드)(비매품)']
매핑 완료: 56 개
샘플: [(0, 1900), (1, 2483), (2, 3351), (3, 3483), (4, 3544)]


In [ ]:
# 기존 CSV들 category_id 변환해서 저장
csv_files = ['wbf_4models', 'wbf_equal', 'wbf_boost_11s', 'wbf_boost_8m', 'wbf_heavy_big']

for name in csv_files:
    df = pd.read_csv(f'{OUTPUT_DIR}/{name}.csv')

    # YOLO 인덱스(1~56) → dl_idx 변환 (저장할 때 +1 했으니 -1 해서 매핑)
    df['category_id'] = df['category_id'].apply(lambda x: idx_to_catid[x - 1])

    out_path = f'{OUTPUT_DIR}/{name}_fixed.csv'
    df.to_csv(out_path, index=False)
    print(f"[{name}] 변환 완료 → category_id 샘플: {df['category_id'].head(3).tolist()}")

[wbf_4models] 변환 완료 → category_id 샘플: [27926, 1900, 16551]
[wbf_equal] 변환 완료 → category_id 샘플: [27926, 1900, 16551]
[wbf_boost_11s] 변환 완료 → category_id 샘플: [27926, 1900, 16551]
[wbf_boost_8m] 변환 완료 → category_id 샘플: [27926, 1900, 16551]
[wbf_heavy_big] 변환 완료 → category_id 샘플: [27926, 1900, 16551]


In [ ]:
df = pd.read_csv(f'{OUTPUT_DIR}/wbf_4models_fixed.csv')
print(df.head(5))
print('category_id 샘플:', df['category_id'].head(10).tolist())

   annotation_id  image_id  category_id  bbox_x  bbox_y  bbox_w  bbox_h  \
0              1         1        27926  595.02  676.52  259.64  478.15   
1              2         1         1900  157.66  251.31  202.41  124.27   
2              3         1        16551  559.87   71.20  393.06  406.40   
3              4         1        29345  170.45  745.15  181.66  291.55   
4              5         1        27733  558.48   71.59  391.42  405.78   

    score  
0  0.9867  
1  0.9616  
2  0.9033  
3  0.3493  
4  0.1418  
category_id 샘플: [27926, 1900, 16551, 29345, 27733, 36637, 27926, 1900, 16551, 29345]


In [ ]:
pred_11s_aug = predict_with_tta(
    f'{BASE}/runs/yolo11s_augmented/weights/best.pt',
    'v11s_aug'
)

[v11s_aug] 3412 rows saved → /content/drive/MyDrive/pill_detection/outputs/pred_v11s_aug.csv


In [ ]:
pred_paths_5 = [pred_8s, pred_8m, pred_11s, pred_11s_aug, pred_11m]
run_wbf_ensemble(pred_paths_5, weights=[0.8, 1.0, 1.2, 1.0, 1.0],
                 out_name='wbf_5models')

[wbf_5models] 4449 rows → /content/drive/MyDrive/pill_detection/outputs/wbf_5models.csv


'/content/drive/MyDrive/pill_detection/outputs/wbf_5models.csv'

In [ ]:
df = pd.read_csv(f'{OUTPUT_DIR}/wbf_5models.csv')
df['category_id'] = df['category_id'].apply(lambda x: idx_to_catid[x - 1])
df.to_csv(f'{OUTPUT_DIR}/wbf_5models_fixed.csv', index=False)
print("변환 완료!")
print(df['category_id'].head(5).tolist())

변환 완료!
[27926, 1900, 16551, 29345, 36637]
